# Phase 1 — Denoising Autoencoder Pretraining
**Continuous Sign Language Translation** · Phase 1 of 2

Trains a `MultiStreamSemanticEncoder` + `StructuredDiffusionDecoder` using a proper DDPM noise schedule with epsilon prediction.

The encoder learns a compact latent `Z [B, T/4, 512]` that will be used in Phase 2 for translation.

**Runtime:** GPU (T4 or better)


In [2]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder.git cslt
%cd cslt
!git pull origin main
!pip install -q -r requirements.txt

fatal: destination path 'cslt' already exists and is not an empty directory.
/content/cslt
From https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


In [4]:
# ── Hugging Face authentication ───────────────────────────────────────
# Set Colab Secret "HF_TOKEN" or paste when prompted. huggingface_hub uses os.environ["HF_TOKEN"]; do not hardcode tokens.
import os
try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (hidden): ")


HF_TOKEN loaded from Colab Secrets ✓


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## ⚙️ Configuration
Adjust these before running. All parameters are passed to the `phase1_pretrain` function via `Phase1Config`.


In [5]:
from config import Phase1Config, MaskingConfig, DDPMConfig

cfg = Phase1Config(
    # Data
    streaming=False,

    # shuffle_buffer is ignored when streaming=False (DataLoader handles it now)

    max_samples=None,             # Full dataset!
    batch_size=32,
    epochs=20,                    # Phase 1 usually needs more time to converge

    # Advanced Denoising logic
    ddpm=DDPMConfig(
        num_timesteps=1000,
        schedule_type="cosine"     # Superior noise schedule
    ),

    # Augmentation logic
    masking=MaskingConfig(
        contrastive_consistency=True, # Enables the NT-Xent contrastive loss
        feature_corruption=True,
        time_span_masking=True,
        whole_part_masking=True
    ),

    # Performance
    mixed_precision=True,          # Essential for GPU speed

    # Checkpointing
    ckpt_dir="/content/phase1_ckpt",
    upload_hf=True,
    hf_repo="mycyrilgoud/data255",
    seed=15179996,
)

print(f"Config: {cfg}")

# try and normalize the latent space
cfg.w_latent_reg = 1.0

Config: Phase1Config(dataset_repo='bdanko/how2sign-landmarks-front-raw-parquet', split='train', max_samples=None, val_max_samples=100, batch_size=32, d_model=384, latent_dim=512, encoder_layers=3, encoder_heads=8, encoder_dropout=0.1, use_part_embeddings=True, ddpm=DDPMConfig(num_timesteps=1000, beta_start=0.0001, beta_end=0.02, schedule_type='cosine'), masking=MaskingConfig(feature_corruption=True, time_span_masking=True, whole_part_masking=True, velocity_reconstruction=True, latent_smoothness=True, contrastive_consistency=True, feature_corruption_prob=0.15, time_span_ratio=0.2, contrastive_weight=0.05), w_masked_pos=1.0, w_masked_vel=1.0, w_full_recon=0.1, w_latent_smooth=0.01, w_contrastive=0.05, w_latent_reg=0.01, epochs=20, lr=0.0001, weight_decay=1e-05, grad_clip=1.0, scheduler='cosine', warmup_steps=100, mixed_precision=True, gradient_accumulation_steps=1, ckpt_dir='/content/phase1_ckpt', hf_repo='mycyrilgoud/data255', upload_hf=True, seed=15179996, log_backend='csv', wandb_proj

## Training
Runs the Phase 1 pretraining loop with DDPM epsilon prediction, configurable masking, train/val splits, and best checkpoint saving.

In [6]:
from phase1_pretrain import train_phase1

train_phase1(cfg)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

...

Train E8:   0%|          | 0/971 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c206418b4c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c206418b4c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Val E8:   0%|          | 0/4 [00:00<?, ?it/s]

Train E9:   0%|          | 0/971 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Upload to Hugging Face (optional)

In [7]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        commit_message=f"Phase 1 checkpoint — {cfg.epochs} epochs, {cfg.max_samples} clips",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...phase1_ckpt/best/model.pt:   0%|          |  113kB / 53.7MB            

  ...e1_ckpt/latest/encoder.pt:   3%|3         | 1.16MB / 35.5MB            

  ...ase1_ckpt/latest/model.pt:   0%|          |  113kB / 53.7MB            

  ...e1_ckpt/best/optimizer.pt:   0%|          | 8.88kB / 92.0MB            

  ..._ckpt/latest/optimizer.pt:   0%|          | 8.88kB / 92.0MB            

  ...ase1_ckpt/best/encoder.pt:  10%|9         | 3.46MB / 35.5MB            

Uploaded to https://huggingface.co/mycyrilgoud/data255 ✓


*(restart session)*

# Phase 2 — FLAN-T5 Translation Fine-tuning
**Continuous Sign Language Translation** · Phase 2 of 2

Loads the frozen `MultiStreamSemanticEncoder` from Phase 1, then fine-tunes a `SignToTextModel` (encoder → attention pooling → adapter → FLAN-T5-small) for utterance-level sign-to-text translation.

> ⚠️ **Run Phase 1 first** — this notebook expects a Phase 1 checkpoint.

**Runtime:** GPU (T4 or better)


In [1]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder.git cslt
%cd cslt
!git pull origin main
!pip install -q -r requirements.txt

fatal: destination path 'cslt' already exists and is not an empty directory.
/content/cslt
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 286 bytes | 286.00 KiB/s, done.
From https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder
 * branch            main       -> FETCH_HEAD
   5a2c951..88a8926  main       -> origin/main
Updating 5a2c951..88a8926
Fast-forward
 phase2_finetune.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


In [3]:
# ── Hugging Face authentication ───────────────────────────────────────
# Set Colab Secret "HF_TOKEN" or paste when prompted. huggingface_hub uses os.environ["HF_TOKEN"]; do not hardcode tokens.
import os
try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (hidden): ")


HF_TOKEN loaded from Colab Secrets ✓


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Download Phase 1 checkpoint (if not local)
If you trained Phase 1 in a previous Colab session, download the encoder weights from HuggingFace.

In [4]:
import os
from huggingface_hub import hf_hub_download

PHASE1_CKPT = "/content/phase1_ckpt"

# Download from HuggingFace if no local checkpoint exists
if not os.path.exists(os.path.join(PHASE1_CKPT, "best", "encoder.pt")):
    os.makedirs(os.path.join(PHASE1_CKPT, "best"), exist_ok=True)
    try:
        encoder_path = hf_hub_download(
            repo_id="mycyrilgoud/data255",
            filename="best/encoder.pt",
            local_dir=PHASE1_CKPT,
        )
        print(f"Encoder downloaded to {encoder_path} ✓")
    except Exception as e:
        print(f"Could not download Phase 1 checkpoint: {e}")
        print("Phase 2 will train encoder from scratch.")
else:
    print(f"Phase 1 checkpoint found at {PHASE1_CKPT}/best/ ✓")

Phase 1 checkpoint found at /content/phase1_ckpt/best/ ✓


## ⚙️ Configuration
Adjust these before running.

In [5]:
from config import Phase2Config

cfg = Phase2Config(
    # Data
    # shuffle_buffer=50000, # This will pull the whole dataset into your 50GB RAM

    streaming=False,

    max_samples=None,            # Train on the full dataset
    batch_size=32,                # Keep batch small to fit T5 + CTC on GPU
    gradient_accumulation_steps=4, # Effective batch size = 16

    # Architecture - Max Implementation
    use_attention_pooling=True,  # Smart temporal aggregation
    use_ctc_head=True,           # Auxiliary temporal alignment loss
    ctc_weight=0.2,              # Balanced weight for CTC loss

    # Training Schedule
    epochs=10,
    warmup_epochs=2,             # 2 epochs of frozen encoder to protect Phase 1 weights
    lr_encoder=5e-6,             # Very slow fine-tuning for encoder
    lr_adapter=5e-5,             # Standard rate for new weights
    lr_t5 = 2e-5,                  # Gentle fine-tuning for T5

    grad_clip = 0.5,          # Tighter clipping

    # Performance & Reliability
    mixed_precision=False,        # Faster training
    num_beams=4,                 # Better generation quality at validation time

    # Infrastructure
    phase1_ckpt="/content/phase1_ckpt", # Point to your new Phase 1
    ckpt_dir="/content/phase2_ckpt",
    upload_hf=True,
    hf_repo="mycyrilgoud/data255",
    seed=15179996,
)

print(f"Config: {cfg}")


Config: Phase2Config(dataset_repo='bdanko/how2sign-landmarks-front-raw-parquet', split='train', max_samples=None, val_max_samples=100, batch_size=32, max_target_length=128, d_model=384, latent_dim=512, encoder_layers=3, encoder_heads=8, encoder_dropout=0.1, use_part_embeddings=True, t5_name='google/flan-t5-base', t5_dim=768, adapter_dropout=0.1, label_smoothing=0.1, use_attention_pooling=True, pool_num_heads=4, epochs=10, warmup_epochs=2, lr_encoder=5e-06, lr_adapter=5e-05, lr_t5=2e-05, weight_decay=1e-05, grad_clip=0.5, scheduler='cosine', warmup_steps=100, mixed_precision=False, gradient_accumulation_steps=4, num_beams=4, max_new_tokens=50, use_ctc_head=True, ctc_weight=0.2, ctc_vocab_size=256, ckpt_dir='/content/phase2_ckpt', phase1_ckpt='/content/phase1_ckpt', hf_repo='mycyrilgoud/data255', upload_hf=True, seed=15179996, log_backend='csv', wandb_project='cslt-phase2', log_every_n_steps=10, shuffle_buffer=1000, streaming=False, smoke_test=False, run_id='5fc2271d')


## Training
Runs the Phase 2 fine-tuning loop with utterance-level supervision, staged unfreezing, and full evaluation (BLEU, ROUGE-L, chrF, exact match).

In [6]:
from phase2_finetune import train_phase2

train_phase2(cfg)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

...


Sample Predictions
  [1] Pred: You're going to have a lot, and it is.
       Ref:  I'll show you.

  [2] Pred: You're going to have a, and you will be in the back of it.
       Ref:  So that is one way, you can just let them go right by.

  [3] Pred: You're going to have a lot, and it is.
       Ref:  If you need to deal with the situation a little bit more it might look something like this.



Train E2:   0%|          | 0/971 [00:00<?, ?it/s]

Val E2:   0%|          | 0/4 [00:00<?, ?it/s]


Sample Predictions
  [1] Pred: I'm going to be in a little bit of it.
       Ref:  I'll show you.

  [2] Pred: I'm going to be in a little bit of it.
       Ref:  So that is one way, you can just let them go right by.

  [3] Pred: I'm going to be in a little bit of it.
       Ref:  If you need to deal with the situation a little bit more it might look something like this.



KeyboardInterrupt: 

## Upload to Hugging Face (optional)

In [7]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        commit_message=f"Phase 2 checkpoint — {cfg.epochs} epochs",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ase2_ckpt/best/encoder.pt: 100%|##########| 35.5MB / 35.5MB            

  ...e2_ckpt/latest/encoder.pt: 100%|##########| 35.5MB / 35.5MB            

  ...ase2_ckpt/latest/model.pt:   3%|3         | 35.4MB / 1.03GB            

...

  ..._ckpt/latest/optimizer.pt:   0%|          |  917kB / 1.76GB            

  ...e2_ckpt/best/optimizer.pt:   0%|          |  921kB / 2.00GB            

  ...ase2_ckpt/best/adapter.pt:   1%|          | 32.9kB / 3.95MB            

  ...e2_ckpt/latest/adapter.pt:   1%|          | 32.9kB / 3.95MB            

Uploaded to https://huggingface.co/mycyrilgoud/data255 ✓


## Quick Inference Test

In [ ]:
from config import InferenceConfig
from inference import run_inference

inf_cfg = InferenceConfig(
    ckpt_dir=cfg.ckpt_dir,
    num_samples=3,
)
run_inference(inf_cfg)